# CS610 Group 1 – NYC Citi Bike demand: full rerun from raw data (forward-chaining ensemble)

This notebook reruns the **whole project end to end on Google Colab**, starting from the raw Citi Bike trip files in
[`data/raw/`](https://github.com/amelia-lee-smu/CS610-Applied-Machine-Learning/tree/main/data/raw):

1. **Preprocessing** – rebuild the station-hour dataset (rental counts, date features, T-2/T-3 lags, T-1 weather),
   then split into train-validation (Jan 2020 – Feb 2021) and test (Mar – Apr 2021).
2. **Base models** – Poisson regression, Random Forest, LightGBM and the PyTorch MLP. Each one makes
   out-of-fold predictions on the 3 expanding-window validation folds, then is refit on Jan 2020 – Feb 2021 and
   predicts the test set.
3. **Ensemble (changed)** – the four combination methods are now chosen with **forward chaining** instead of
   leave-one-fold-out, so the ensemble weights are **only ever learned from earlier folds**:

| Validation fold scored | Ensemble weights learned from | Rows |
|---|---|---|
| F1 Sep–Oct 2020 | – (no earlier fold, cannot be scored) | 74,664 |
| **F2 Nov–Dec 2020** | **F1** | 74,664 |
| **F3 Jan–Feb 2021** | **F1 + F2** | 72,216 |
| Test Mar–Apr 2021 (final) | F1 + F2 + F3 | 74,409 |

**Why:** the old leave-one-fold-out scored F1 with weights learned on F2 + F3 and F2 with weights learned on F1 + F3,
i.e. it used *future* months to score *past* months – information leakage. Forward chaining removes it.

**Runtime** (default Colab CPU runtime): ~10–20 min with `RUN_OPTUNA = False` (uses the team's tuned
hyperparameters). With `RUN_OPTUNA = True` the full Optuna searches take several hours (Random Forest is the slowest).

Run all cells from top to bottom (*Runtime → Run all*).

## 0. Settings

In [ ]:
# --- main switches ---------------------------------------------------------------------------------------------
RUN_OPTUNA = False          # False: use the tuned hyperparameters found by the team (fast)
                            # True:  rerun the Optuna searches (same search spaces / trials as the original notebooks)
N_TRIALS = {'poisson_regression': 30, 'random_forest': 20, 'lightgbm': 30, 'neural_network': 20}

USE_WEATHER_API = False     # False: use the Open-Meteo weather the team already downloaded
                            #        (data/intermediate_processing/historical_weather_features.csv) - identical numbers
                            # True:  call the Open-Meteo archive API again for all 51 stations (~3-4 min, rate-limited)

MODELS_TO_RUN = ['poisson_regression', 'random_forest', 'lightgbm', 'neural_network']

SAVE_TO_DRIVE = True        # copy the outputs to Google Drive (MyDrive/CS610/rerun_forward_chaining/) at the end

# tuned hyperparameters (read from the team's saved models in models/*.pkl)
BEST_PARAMS = {
    'poisson_regression': {'alpha': 4.395874560753534e-05, 'solver': 'lbfgs'},
    'random_forest': {'n_estimators': 200, 'max_depth': 24, 'min_samples_leaf': 12,
                      'max_features': 0.9663483218220689, 'max_samples': 0.30361603283123195,
                      'criterion': 'squared_error'},
    'lightgbm': {'learning_rate': 0.04248968358402182, 'n_estimators': 700, 'num_leaves': 230,
                 'min_child_samples': 38, 'subsample': 0.8368552281836176,
                 'colsample_bytree': 0.5384209448817046, 'reg_lambda': 0.04841646666898234},
    'neural_network': {'station_dim': 4, 'hidden_size': 32, 'n_layers': 3, 'dropout': 0.011121331424011031,
                       'learning_rate': 0.002866757342953708, 'weight_decay': 1.5606637983055393e-06,
                       'epochs': 20, 'batch_size': 1024},
}

## 1. Setup: get the repo and the packages

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/amelia-lee-smu/CS610-Applied-Machine-Learning.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    os.chdir('/content')
    if not os.path.exists('CS610-Applied-Machine-Learning'):
        subprocess.run(['git', 'clone', '-q', REPO_URL], check = True)
    os.chdir('/content/CS610-Applied-Machine-Learning')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna', 'holidays',
                    'openmeteo-requests', 'requests-cache', 'retry-requests'], check = True)

REPO_DIR = os.getcwd()
sys.path.insert(0, REPO_DIR)
print('Working in', REPO_DIR)
print(subprocess.run(['git', 'log', '-1', '--format=%h %ad %an: %s', '--date=short'],
                     capture_output = True, text = True).stdout)

In [ ]:
import datetime as dt
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import sklearn, lightgbm, torch

from utilities.values import *
from utilities.utilities import *
from utilities.training import cross_validate, get_out_of_fold_predictions, evaluate

if hasattr(pd.errors, 'SettingWithCopyWarning'):   # pandas < 3 warns when the shared code adds a column to a slice
    warnings.filterwarnings('ignore', category = pd.errors.SettingWithCopyWarning)
warnings.filterwarnings('ignore', category = DeprecationWarning)
pd.set_option('display.width', 160)

DATA_DIR = 'data/'
MODEL_DIR = 'models/'
OUT_DIR = 'outputs_rerun/'           # everything produced by this notebook also goes here
os.makedirs(OUT_DIR, exist_ok = True)

print('pandas', pd.__version__, '| scikit-learn', sklearn.__version__, '| lightgbm', lightgbm.__version__,
      '| torch', torch.__version__, '| optuna', optuna.__version__)
print('CPU cores:', os.cpu_count())

# the expanding-window folds must be in time order with no overlap: every model trained for fold k only sees
# data that ends before fold k starts, so its out-of-fold predictions never use the future
for k, (train_start, train_end, val_start, val_end) in enumerate(VALIDATION_FOLDS, start = 1):
    assert train_end < val_start, f'fold {k}: training data overlaps its validation months'
    if k > 1:
        assert VALIDATION_FOLDS[k - 2][3] < val_start, f'fold {k - 1} and fold {k} overlap'
    print(f'Fold {k}: train {train_start} → {train_end} | validate {val_start} → {val_end}')

## 2. Preprocessing from the raw trip files

Same steps as `notebooks/data_preprocessing.ipynb`.

### 2a. Merge the raw monthly files (Jan 2020 – Apr 2021)

In [ ]:
from utilities.preprocessor import read_all_files_in_dir, DateFeatureExtractor

raw_data_dir_paths = [DATA_DIR + 'raw/2020', DATA_DIR + 'raw/2021']
for d in raw_data_dir_paths:
    print(d, sorted(f.name for f in Path(d).iterdir()))

full_dataset_df = read_all_files_in_dir(raw_data_dir_paths)
full_dataset_df['starttime'] = pd.to_datetime(full_dataset_df['starttime'])

# the Feb-Apr 2021 files are in UTC, while all earlier files are in New York local time, so convert them
is_utc = full_dataset_df['starttime'] >= '2021-02-01'
full_dataset_df.loc[is_utc, 'starttime'] = (
    full_dataset_df.loc[is_utc, 'starttime']
    .dt.tz_localize('UTC')
    .dt.tz_convert(NYC_TIMEZONE)
    .dt.tz_localize(None)
)

# remove duplicate trips (corrected Feb 2021 file overlaps with Jan 2021 files)
full_dataset_df = full_dataset_df.drop_duplicates(subset = ['bikeid', 'starttime', 'start station name'])
full_dataset_df = full_dataset_df.sort_values(['starttime', 'stoptime'])

print('Total number of trips:', f'{len(full_dataset_df):,}')
print('Data period:', full_dataset_df['starttime'].min(), '→', full_dataset_df['starttime'].max())

### 2b. Rental counts on a complete station × date × hour grid, date features and T-2 / T-3 lags

In [ ]:
extractor = DateFeatureExtractor()

# remove the depot, which only has 1 rental and is not an actual station
full_dataset_df = full_dataset_df[full_dataset_df[STATION] != 'JCBS Depot']
full_dataset_df[DATE] = full_dataset_df['starttime'].dt.date
full_dataset_df[HOUR_OF_DAY] = full_dataset_df['starttime'].dt.hour

# count rentals for each station-date-hour (only includes those with at least 1 rental)
rental_counts_df = full_dataset_df.groupby([STATION, DATE, HOUR_OF_DAY]).size().reset_index(name = NUM_RENTALS)

# every station-date-hour gets a row, even with 0 rentals
stations = rental_counts_df[STATION].unique()
dates = pd.date_range('2020-01-01', '2021-04-30', freq = 'D').date
full_rental_counts_df = pd.MultiIndex.from_product(
    [stations, dates, range(24)], names = [STATION, DATE, HOUR_OF_DAY]
).to_frame(index = False)

# the last raw file ends at 19:59 on 30 Apr 2021 (New York time), so there is no data for 8pm-11pm that day
no_data = (full_rental_counts_df[DATE] == dt.date(2021, 4, 30)) & (full_rental_counts_df[HOUR_OF_DAY] >= 20)
full_rental_counts_df = full_rental_counts_df[~no_data]

full_rental_counts_df = full_rental_counts_df.merge(rental_counts_df, on = [STATION, DATE, HOUR_OF_DAY], how = 'left')
full_rental_counts_df[NUM_RENTALS] = full_rental_counts_df[NUM_RENTALS].fillna(0).astype(int)

# station coordinates (needed for the weather API)
station_locations_df = full_dataset_df.groupby(STATION)[[LATITUDE, LONGITUDE]].first().reset_index()
full_rental_counts_df = full_rental_counts_df.merge(station_locations_df, on = STATION, how = 'left')

# date features
full_rental_counts_df[WEEK_OF_YEAR] = full_rental_counts_df[DATE].apply(extractor.to_week_of_year)
full_rental_counts_df[SINE_WEEK_OF_YEAR] = extractor.to_sine_week_of_year(full_rental_counts_df[WEEK_OF_YEAR])
full_rental_counts_df[COSINE_WEEK_OF_YEAR] = extractor.to_cosine_week_of_year(full_rental_counts_df[WEEK_OF_YEAR])
full_rental_counts_df[DAY_OF_WEEK] = full_rental_counts_df[DATE].apply(extractor.to_day_of_week)
full_rental_counts_df[IS_PUBLIC_HOLIDAY] = full_rental_counts_df[DATE].apply(extractor.to_is_public_holiday)
full_rental_counts_df = full_rental_counts_df.drop(columns = [WEEK_OF_YEAR])
full_rental_counts_df = full_rental_counts_df.sort_values([STATION, DATE, HOUR_OF_DAY]).reset_index(drop = True)

# lag features: predicting hour T at T-1, so the latest known rental counts are T-2 and T-3
rental_timestamp = pd.to_datetime(full_rental_counts_df[DATE]) + pd.to_timedelta(full_rental_counts_df[HOUR_OF_DAY], unit = 'h')
rentals_lookup = (full_rental_counts_df.assign(timestamp = rental_timestamp)
                  .groupby([STATION, 'timestamp'])[NUM_RENTALS].sum())
data_start = rental_timestamp.min()
for lag in [2, 3]:
    lag_timestamp = rental_timestamp - pd.Timedelta(hours = lag)
    lag_keys = pd.MultiIndex.from_arrays([full_rental_counts_df[STATION], lag_timestamp])
    lag_values = pd.Series(rentals_lookup.reindex(lag_keys).fillna(0).to_numpy(), index = full_rental_counts_df.index)
    full_rental_counts_df[f'{NUM_RENTALS}_t-{lag}'] = lag_values.where(lag_timestamp >= data_start, 0)

print(f'{len(full_rental_counts_df):,} station-hours on the grid, '
      f'{(full_rental_counts_df[NUM_RENTALS] > 0).sum():,} with at least 1 rental, {len(stations)} stations')
full_rental_counts_df.head()

### 2c. Hourly weather (Open-Meteo archive)

In [ ]:
weather_cache = DATA_DIR + 'intermediate_processing/historical_weather_features.csv'

if USE_WEATHER_API:
    from utilities.preprocessor import WeatherFeatureExtractor
    weather_search_df = full_rental_counts_df[[STATION, LATITUDE, LONGITUDE, DATE]].drop_duplicates()
    start_date = weather_search_df[DATE].min() - dt.timedelta(days = 1)  # 1 extra day so the first hour has T-1 weather
    end_date = weather_search_df[DATE].max()
    weather_extractor = WeatherFeatureExtractor(start_date, end_date)
    parts = []
    for i, station in enumerate(sorted(set(weather_search_df[STATION])), start = 1):
        loc = weather_search_df[weather_search_df[STATION] == station].iloc[0]
        parts.append(weather_extractor.get_hourly_weather(station, loc[LATITUDE], loc[LONGITUDE]))
        if i % 20 == 0:   # stay under the API rate limit
            print(f'{i} stations done, sleeping for 60s...')
            time.sleep(60)
    weather_df = pd.concat(parts, ignore_index = True)
    weather_df.to_csv(OUT_DIR + 'historical_weather_features.csv', index = False)
    print('Weather downloaded from the Open-Meteo API')
else:
    weather_df = pd.read_csv(weather_cache)
    print('Weather read from', weather_cache)

print(f'{len(weather_df):,} weather rows for {weather_df[STATION].nunique()} stations')

### 2d. Merge the weather at T-1, select features, split train-validation / test

In [ ]:
# write and read back through csv, exactly like the original notebook (dates become 'YYYY-MM-DD' strings)
full_rental_counts_df.to_csv(OUT_DIR + 'full_rental_counts.csv', index = False)
full_rental_counts_df = pd.read_csv(OUT_DIR + 'full_rental_counts.csv')
weather_df[DATE] = pd.to_datetime(weather_df[DATE]).dt.strftime('%Y-%m-%d')

merge_columns = [STATION, DATE, HOUR_OF_DAY]
# one weather row per station-date-hour (drops the repeated hour when daylight saving ends)
weather_to_merge_df = (weather_df.drop(columns = [LATITUDE, LONGITUDE])
                       .drop_duplicates(subset = merge_columns, keep = 'first'))

# lag the weather by 1 hour: the weather recorded at T-1 is used to predict hour T
lagged_timestamp = pd.to_datetime(weather_to_merge_df[DATE]) + pd.to_timedelta(weather_to_merge_df[HOUR_OF_DAY] + 1, unit = 'h')
weather_to_merge_df[DATE] = lagged_timestamp.dt.strftime('%Y-%m-%d')
weather_to_merge_df[HOUR_OF_DAY] = lagged_timestamp.dt.hour

full_features_df = full_rental_counts_df.merge(weather_to_merge_df, on = merge_columns, how = 'left', validate = 'many_to_one')

# 2am does not exist on the days the clocks go forward, so 3am has no T-1 weather: use the previous hour (1am)
weather_cols = [c for c in weather_to_merge_df.columns if c not in merge_columns]
full_features_df = full_features_df.sort_values([STATION, DATE, HOUR_OF_DAY])
full_features_df[weather_cols] = full_features_df.groupby(STATION)[weather_cols].ffill()
nonexistent_hour = full_features_df[DATE].isin(['2020-03-08', '2021-03-14']) & (full_features_df[HOUR_OF_DAY] == 2)
full_features_df = full_features_df[~nonexistent_hour].drop(columns = [LATITUDE, LONGITUDE])

# feature selection (see eda_weather_feature_selection.ipynb)
full_features_df = full_features_df.drop(columns = [RAIN, SNOWFALL, WIND_GUSTS, CLOUD_COVER_LOW, CLOUD_COVER_HIGH])
assert full_features_df[NUMERICAL_COLUMNS].notna().all().all(), 'missing feature values'

column_order = [DATE, NUM_RENTALS] + CATEGORICAL_COLUMNS + NUMERICAL_COLUMNS + OTHER_COLUMNS
train_validation_df = full_features_df[full_features_df[DATE] <= '2021-02-28'][column_order].reset_index(drop = True)
test_df = full_features_df[full_features_df[DATE] >= '2021-03-01'][column_order].reset_index(drop = True)

save_to_parquet_file(train_validation_df, OUT_DIR + 'processed/train_validation.parquet')
save_to_parquet_file(test_df, OUT_DIR + 'processed/test.parquet')
print(f'Train-validation (Jan 2020 - Feb 2021): {len(train_validation_df):,} rows   (slides: 520,149)')
print(f'Test (Mar - Apr 2021):                  {len(test_df):,} rows    (slides: 74,409)')

### 2e. Check: does the rebuilt data match the team's processed files?

In [ ]:
def compare_with_repo(rebuilt_df, repo_file):
    repo_df = read_parquet_file(repo_file)
    keys = [DATE, STATION, HOUR_OF_DAY]
    a = rebuilt_df.assign(**{DATE: rebuilt_df[DATE].astype(str)}).sort_values(keys).reset_index(drop = True)
    b = repo_df.assign(**{DATE: repo_df[DATE].astype(str)}).sort_values(keys).reset_index(drop = True)
    if len(a) != len(b):
        return f'{repo_file}: different number of rows ({len(a):,} rebuilt vs {len(b):,} in repo)'
    assert (a[keys].astype(str).values == b[keys].astype(str).values).all(), 'rows differ'
    diffs = {c: float(np.abs(a[c].astype(float) - b[c].astype(float)).max())
             for c in [NUM_RENTALS, DAY_OF_WEEK] + NUMERICAL_COLUMNS + OTHER_COLUMNS}
    worst = max(diffs, key = diffs.get)
    return f'{repo_file}: same {len(a):,} rows; largest difference in any column = {diffs[worst]:.2e} ({worst})'

for rebuilt, name in [(train_validation_df, 'train_validation'), (test_df, 'test')]:
    repo_file = DATA_DIR + f'processed/{name}.parquet'
    if Path(repo_file).exists():
        print(compare_with_repo(rebuilt, repo_file))

## 3. Base models

For every model:
1. *(optional, `RUN_OPTUNA = True`)* Optuna search on the mean 3-fold validation Poisson deviance;
2. **out-of-fold predictions** on F1, F2, F3 – each from a new model trained only on the data before that fold;
3. a final model trained on Jan 2020 – Feb 2021, which predicts the test set.

In [ ]:
from utilities.poisson_regression import PoissonRegressionModel
from utilities.random_forest import RandomForestModel
from utilities.lightgbm import LightGBMModel
from utilities.neural_network import NeuralNetworkModel

MODEL_CLASSES = {
    'poisson_regression': PoissonRegressionModel,
    'random_forest': RandomForestModel,
    'lightgbm': LightGBMModel,
    'neural_network': NeuralNetworkModel,
}

# Optuna search spaces (the same as the team's training notebooks)
SEARCH_SPACES = {
    'poisson_regression': lambda t: {
        'alpha': t.suggest_float('alpha', 1e-6, 1.0, log = True),
        'solver': t.suggest_categorical('solver', ['lbfgs', 'newton-cholesky']),
    },
    'random_forest': lambda t: {
        'n_estimators': t.suggest_int('n_estimators', 50, 200, step = 50),
        'max_depth': t.suggest_int('max_depth', 8, 30),
        'min_samples_leaf': t.suggest_int('min_samples_leaf', 5, 200, log = True),
        'max_features': t.suggest_float('max_features', 0.2, 1.0),
        'max_samples': t.suggest_float('max_samples', 0.1, 0.4),
        'criterion': t.suggest_categorical('criterion', ['poisson', 'squared_error']),
    },
    'lightgbm': lambda t: {
        'learning_rate': t.suggest_float('learning_rate', 0.01, 0.2, log = True),
        'n_estimators': t.suggest_int('n_estimators', 100, 800, step = 50),
        'num_leaves': t.suggest_int('num_leaves', 15, 255, log = True),
        'min_child_samples': t.suggest_int('min_child_samples', 10, 300, log = True),
        'subsample': t.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': t.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda': t.suggest_float('reg_lambda', 1e-3, 10.0, log = True),
    },
    'neural_network': lambda t: {
        'station_dim': t.suggest_categorical('station_dim', [4, 8, 16]),
        'hidden_size': t.suggest_categorical('hidden_size', [32, 64, 128]),
        'n_layers': t.suggest_int('n_layers', 1, 3),
        'dropout': t.suggest_float('dropout', 0.0, 0.3),
        'learning_rate': t.suggest_float('learning_rate', 1e-3, 1e-2, log = True),
        'weight_decay': t.suggest_float('weight_decay', 1e-6, 1e-3, log = True),
        'epochs': t.suggest_int('epochs', 8, 20),
        'batch_size': t.suggest_categorical('batch_size', [1024, 2048, 4096]),
    },
}


def tune(model_name):
    '''Optuna search: lowest mean Poisson deviance over the 3 validation folds.'''
    def objective(trial):
        scores = cross_validate(MODEL_CLASSES[model_name], SEARCH_SPACES[model_name](trial), train_validation_df)
        mean = scores[['MAE', 'RMSE', 'poisson_deviance']].mean()
        trial.set_user_attr('MAE', mean['MAE'])
        trial.set_user_attr('RMSE', mean['RMSE'])
        return mean['poisson_deviance']
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler(seed = OPTUNA_SEED))
    study.optimize(objective, n_trials = N_TRIALS[model_name], show_progress_bar = True)
    print('Best hyperparameters:', study.best_params, '| mean deviance', round(study.best_value, 4))
    return study.best_params


def run_model(model_name):
    started = time.time()
    params = tune(model_name) if RUN_OPTUNA else BEST_PARAMS[model_name]
    print(f'{model_name}: {params}')

    # out-of-fold predictions on each validation fold (each fold's model only sees earlier data)
    for fold_number, fold_df in enumerate(
            get_out_of_fold_predictions(MODEL_CLASSES[model_name], params, train_validation_df), start = 1):
        save_to_parquet_file(fold_df, OUT_DIR + f'predictions/validation_{model_name}_fold{fold_number}.parquet')

    # final model on all of Jan 2020 - Feb 2021, then the test set
    model = MODEL_CLASSES[model_name](**params)
    model.fit(train_validation_df)
    predicted_test_df = test_df.copy()
    predicted_test_df[PREDICTION] = model.predict(predicted_test_df)
    model.save_model(OUT_DIR + f'models/{model_name}_model.pkl')
    save_to_parquet_file(predicted_test_df, OUT_DIR + f'predictions/test_{model_name}.parquet')

    test_scores = evaluate(model, predicted_test_df)
    print(f'{model_name} done in {(time.time() - started) / 60:.1f} min | test (report only): '
          + ', '.join(f'{k} {v:.4f}' for k, v in test_scores.items()))
    return params


USED_PARAMS = {}

### 3a. Poisson regression

In [ ]:
if 'poisson_regression' in MODELS_TO_RUN:
    USED_PARAMS['poisson_regression'] = run_model('poisson_regression')

### 3b. Random forest *(slowest model, ~5–10 min on the free CPU runtime)*

In [ ]:
if 'random_forest' in MODELS_TO_RUN:
    USED_PARAMS['random_forest'] = run_model('random_forest')

### 3c. LightGBM

In [ ]:
if 'lightgbm' in MODELS_TO_RUN:
    USED_PARAMS['lightgbm'] = run_model('lightgbm')

### 3d. Neural network (PyTorch MLP)

In [ ]:
if 'neural_network' in MODELS_TO_RUN:
    USED_PARAMS['neural_network'] = run_model('neural_network')

## 4. Ensemble with forward chaining (no information leakage)

The base models are not retrained here: the ensemble only learns how to combine their saved predictions.

**Forward chaining:**
- **Score fold 2 (Nov–Dec 2020) with weights learned on F1** (Sep–Oct 2020)
- **Score fold 3 (Jan–Feb 2021) with weights learned on F1 + F2** (Sep–Dec 2020)
- F1 has no earlier fold, so it cannot be scored.

The method with the lowest mean forward-chaining Poisson deviance (folds 2–3) is chosen. Its final weights are then
learned on F1 + F2 + F3 (all before March 2021) and applied to the test set.

In [ ]:
from utilities.ensemble import EnsembleModel, ENSEMBLE_METHODS, KEY_COLUMNS, read_predictions, score

MODELS = [m for m in ['poisson_regression', 'lightgbm', 'neural_network', 'random_forest'] if m in MODELS_TO_RUN]
NUM_FOLDS = len(VALIDATION_FOLDS)

validation_df = pd.concat([
    read_predictions({m: OUT_DIR + f'predictions/validation_{m}_fold{n}.parquet' for m in MODELS})
    for n in range(1, NUM_FOLDS + 1)
], ignore_index = True)
ens_test_df = read_predictions({m: OUT_DIR + f'predictions/test_{m}.parquet' for m in MODELS})

fold_info = validation_df.assign(d = pd.to_datetime(validation_df[DATE])).groupby('fold')['d'].agg(['min', 'max', 'size'])
fold_info.columns = ['first day', 'last day', 'station-hours']
print('Out-of-fold predictions of the base models (the ensemble training data):')
display(fold_info)
print(f'test: {len(ens_test_df):,} station-hours')


def check_no_time_leakage(df):
    '''Validation folds must be in time order and must not overlap.'''
    ranges = pd.to_datetime(df[DATE]).groupby(df['fold']).agg(['min', 'max']).sort_index()
    for earlier, later in zip(ranges.index[:-1], ranges.index[1:]):
        assert ranges.loc[earlier, 'max'] < ranges.loc[later, 'min'], f'fold {earlier} overlaps fold {later}'


def forward_chaining(df, model_names, method):
    '''
    Time-safe validation of an ensemble method: the combination is learned ONLY on the folds before the scored fold.
        fold 2  <-  weights learned on fold 1
        fold 3  <-  weights learned on folds 1 + 2
    Returns (scores per scored fold, the out-of-fold ensemble predictions).
    '''
    check_no_time_leakage(df)
    rows, predictions = [], []
    folds = sorted(df['fold'].unique())
    for fold_number in folds[1:]:
        train_df, held_out_df = df[df['fold'] < fold_number], df[df['fold'] == fold_number]
        assert pd.to_datetime(train_df[DATE]).max() < pd.to_datetime(held_out_df[DATE]).min(), 'time leakage'
        ensemble = EnsembleModel(model_names, method).fit(train_df)
        predicted = ensemble.predict(held_out_df)
        rows.append({'method': f'ensemble: {method}', 'fold': fold_number,
                     'weights learned on': ' + '.join(f'F{f}' for f in sorted(train_df['fold'].unique())),
                     'train rows': len(train_df), 'scored rows': len(held_out_df),
                     **score(held_out_df[NUM_RENTALS], predicted),
                     **{f'w_{m}': w for m, w in zip(model_names, ensemble.weights)}})
        predictions.append(held_out_df[KEY_COLUMNS + [NUM_RENTALS, 'fold']].assign(**{PREDICTION: predicted}))
    return pd.DataFrame(rows), pd.concat(predictions, ignore_index = True)


def leave_one_fold_out(df, model_names, method):
    '''OLD method, NOT time-safe (scores F1 and F2 with weights learned partly on later folds). Comparison only.'''
    rows = []
    for fold_number in sorted(df['fold'].unique()):
        train_df, held_out_df = df[df['fold'] != fold_number], df[df['fold'] == fold_number]
        ensemble = EnsembleModel(model_names, method).fit(train_df)
        rows.append({'method': f'ensemble: {method}', 'fold': fold_number,
                     **score(held_out_df[NUM_RENTALS], ensemble.predict(held_out_df))})
    return pd.DataFrame(rows)

In [ ]:
fc_results = {method: forward_chaining(validation_df, MODELS, method) for method in ENSEMBLE_METHODS}
fc_scores_df = pd.concat([r[0] for r in fc_results.values()], ignore_index = True)

# the base models on exactly the same folds (2 and 3), so the comparison is like-for-like
base_rows = []
for model_name in MODELS:
    for fold_number, fold_df in validation_df[validation_df['fold'] > 1].groupby('fold'):
        base_rows.append({'method': model_name, 'fold': fold_number, **score(fold_df[NUM_RENTALS], fold_df[model_name])})
base_scores_df = pd.DataFrame(base_rows)

all_fc_df = pd.concat([fc_scores_df, base_scores_df], ignore_index = True)
fc_summary_df = (all_fc_df.groupby('method')[['poisson_deviance', 'MAE', 'RMSE']].mean()
                 .sort_values('poisson_deviance'))

print('Forward chaining: mean of fold 2 (weights from F1) and fold 3 (weights from F1+F2), lower is better')
display(fc_summary_df.round(4))

print('Poisson deviance per scored fold:')
display(all_fc_df.pivot(index = 'method', columns = 'fold', values = 'poisson_deviance')
        .rename(columns = {2: 'F2 Nov–Dec 2020', 3: 'F3 Jan–Feb 2021'})
        .assign(mean = lambda d: d.mean(axis = 1)).sort_values('mean').round(4))

print('Weights used to score each fold (learned only on earlier folds):')
display(fc_scores_df[['method', 'fold', 'weights learned on', 'train rows', 'scored rows']
                     + [f'w_{m}' for m in MODELS]].round(3))

BEST_METHOD = fc_scores_df.groupby('method')['poisson_deviance'].mean().idxmin().replace('ensemble: ', '')
print('Chosen method (lowest forward-chaining deviance):', BEST_METHOD, '-', ENSEMBLE_METHODS[BEST_METHOD])

### How much did the old leave-one-fold-out flatter the scores?

Same methods, old (leaky) scoring. The leaky numbers average over all 3 folds, so they are not directly comparable
with the base models on folds 2–3; the point is the **ranking** and whether the chosen method would change.

In [ ]:
loo_df = pd.concat([leave_one_fold_out(validation_df, MODELS, m) for m in ENSEMBLE_METHODS], ignore_index = True)
leaky_vs_safe = pd.DataFrame({
    'leave-one-fold-out (old, leaky), folds 1-3': loo_df.groupby('method')['poisson_deviance'].mean(),
    'leave-one-fold-out, folds 2-3 only': loo_df[loo_df['fold'] > 1].groupby('method')['poisson_deviance'].mean(),
    'forward chaining (new), folds 2-3': fc_scores_df.groupby('method')['poisson_deviance'].mean(),
}).sort_values('forward chaining (new), folds 2-3')
display(leaky_vs_safe.round(4))
old_choice = leaky_vs_safe.iloc[:, 0].idxmin().replace('ensemble: ', '')
print(f'Old method would choose: {old_choice} | forward chaining chooses: {BEST_METHOD}')

## 5. Final ensemble and test results

The chosen method learns its final weights on **F1 + F2 + F3** (Sep 2020 – Feb 2021, all before the test months), then
combines the base models' test predictions. The test set (Mar – Apr 2021) is scored once, for reporting only.

In [ ]:
final_ensemble = EnsembleModel(MODELS, method = BEST_METHOD).fit(validation_df)
print(f'Final method: {BEST_METHOD}, weights learned on folds {[int(f) for f in sorted(validation_df["fold"].unique())]}')
display(final_ensemble.weights_table().round(4).to_frame())

final_weights_df = pd.concat({m: EnsembleModel(MODELS, m).fit(validation_df).weights_table().rename(None)
                              for m in ENSEMBLE_METHODS}, axis = 1)
print('Final weights of every method (stacking: coefficients on the log scale, intercept not shown):')
display(final_weights_df.round(4))

ens_test_df[PREDICTION] = final_ensemble.predict(ens_test_df)
final_ensemble.save_model(OUT_DIR + 'models/ensemble_model.pkl')
save_to_parquet_file(ens_test_df, OUT_DIR + 'predictions/test_ensemble.parquet')

In [ ]:
rows = [{'method': m, **score(ens_test_df[NUM_RENTALS], ens_test_df[m])} for m in MODELS]
for method in ENSEMBLE_METHODS:
    fitted = EnsembleModel(MODELS, method).fit(validation_df)
    rows.append({'method': f'ensemble: {method}' + (' (chosen)' if method == BEST_METHOD else ''),
                 **score(ens_test_df[NUM_RENTALS], fitted.predict(ens_test_df))})
test_scores_df = pd.DataFrame(rows).set_index('method').sort_values('poisson_deviance')
print('Test set, Mar – Apr 2021 (lower is better; NOT used for any choice):')
display(test_scores_df.round(4))

### Diagnostic: hours with at least one rental

72% of station-hours have no rentals, which flatters every all-rows score. These rows score the same models only
where rentals actually happened – on the validation folds 2–3 (forward-chained ensemble) and on the test set.

In [ ]:
fc_pred_chosen = fc_results[BEST_METHOD][1]
val23 = validation_df[validation_df['fold'] > 1].merge(
    fc_pred_chosen[KEY_COLUMNS + [PREDICTION]], on = KEY_COLUMNS, how = 'left', validate = 'one_to_one')
rows = []
for name in MODELS + [PREDICTION]:
    label = f'ensemble: {BEST_METHOD}' if name == PREDICTION else name
    for subset_name, frame in [('validation F2-F3', val23), ('test', ens_test_df)]:
        positive = frame[frame[NUM_RENTALS] > 0]
        rows.append({'method': label, 'data': subset_name, 'rows': len(positive),
                     **score(positive[NUM_RENTALS], positive[name])})
rental_hours_df = pd.DataFrame(rows).pivot(index = 'method', columns = 'data', values = ['MAE', 'RMSE', 'poisson_deviance'])
display(rental_hours_df.round(3))

In [ ]:
daily_df = ens_test_df.groupby(DATE)[[NUM_RENTALS, PREDICTION] + MODELS].sum()
daily_df.index = pd.to_datetime(daily_df.index)
fig, ax = plt.subplots(figsize = (12, 4.5))
ax.plot(daily_df.index, daily_df[NUM_RENTALS], color = 'black', linewidth = 2, label = 'actual')
ax.plot(daily_df.index, daily_df[PREDICTION], color = '#eb6834', linewidth = 2, label = f'ensemble ({BEST_METHOD})')
for m in MODELS:
    ax.plot(daily_df.index, daily_df[m], linewidth = 1, alpha = 0.45, label = m)
ax.set_ylabel('rentals per day (all stations)')
ax.set_title('Test period: daily rentals, actual vs predicted', loc = 'left')
ax.legend(ncol = 3, fontsize = 9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR + 'test_daily_rentals.png', dpi = 120)
plt.show()

## 6. Save the results

Colab deletes its files when the session ends. Everything is in `outputs_rerun/` (same folder structure as the repo),
zipped and downloaded, and optionally copied to Google Drive.

In [ ]:
fc_summary_df.to_csv(OUT_DIR + 'ensemble_validation_scores_forward_chaining.csv')
fc_scores_df.to_csv(OUT_DIR + 'ensemble_forward_chaining_per_fold.csv', index = False)
leaky_vs_safe.to_csv(OUT_DIR + 'ensemble_leaky_vs_forward_chaining.csv')
test_scores_df.to_csv(OUT_DIR + 'ensemble_test_scores.csv')
final_weights_df.to_csv(OUT_DIR + 'ensemble_weights.csv')
rental_hours_df.to_csv(OUT_DIR + 'scores_on_rental_hours.csv')
pd.Series({m: str(p) for m, p in USED_PARAMS.items()}, name = 'hyperparameters').to_csv(OUT_DIR + 'hyperparameters_used.csv')

zip_path = shutil.make_archive('/tmp/cs610_rerun_outputs', 'zip', OUT_DIR)
print('Zipped:', zip_path, f'({os.path.getsize(zip_path) / 1e6:.1f} MB)')

if IN_COLAB:
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_dir = '/content/drive/MyDrive/CS610/rerun_forward_chaining'
        os.makedirs(drive_dir, exist_ok = True)
        shutil.copy(zip_path, drive_dir)
        print('Copied to', drive_dir)
    from google.colab import files
    files.download(zip_path)